# 02 — Data Cleaning & Preprocessing

This notebook applies a comprehensive cleaning pipeline to the raw financial data:

1. Standardize column names (lowercase, no spaces)
2. Handle missing values
3. Convert dates → datetime, prices → numeric
4. Remove `$` symbols from price columns
5. Create `daily_return` column (% change of close price)
6. Export cleaned datasets to `data/processed/`

---

In [ ]:
import pandas as pd
import numpy as np
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Add src/ to path so we can import our modules
sys.path.insert(0, os.path.join('..', 'src'))

from data_processing import (
 load_raw_stock, clean_stock, load_and_clean_all_stocks,
 load_raw_mutual_funds, clean_mutual_funds, save_processed, build_all
)

RAW_DIR = os.path.join('..', 'data', 'raw')
PROC_DIR = os.path.join('..', 'data', 'processed')
TICKERS = ['AAPL', 'MSFT', 'GOOG']

## 1. Stock Data Cleaning — Step by Step

In [ ]:
# Load a raw stock file to demonstrate the cleaning process
raw_aapl = load_raw_stock('AAPL', RAW_DIR)

print('BEFORE CLEANING')
print('=' * 50)
print(f'Columns: {raw_aapl.columns.tolist()}')
print(f'Dtypes:\n{raw_aapl.dtypes}')
print(f'\nSample values (first 3 rows):')
display(raw_aapl.head(3))

In [ ]:
# Apply our cleaning function
clean_aapl = clean_stock(raw_aapl, 'AAPL')

print('AFTER CLEANING')
print('=' * 50)
print(f'Columns: {clean_aapl.columns.tolist()}')
print(f'Dtypes:\n{clean_aapl.dtypes}')
print(f'\nSample values (first 5 rows):')
display(clean_aapl.head(5))

In [ ]:
# Verify cleaning transformations
print('Cleaning Verification')
print('=' * 50)
print(f'✓ Column names lowercase : {all(c == c.lower() for c in clean_aapl.columns)}')
print(f'✓ Date is datetime   : {pd.api.types.is_datetime64_any_dtype(clean_aapl["date"])}')
print(f'✓ Close is numeric   : {pd.api.types.is_numeric_dtype(clean_aapl["close"])}')
print(f'✓ No $ in close    : {not clean_aapl["close"].astype(str).str.contains(r"\$").any()}')
print(f'✓ daily_return exists  : {"daily_return" in clean_aapl.columns}')
print(f'✓ Sorted by date (asc)  : {clean_aapl["date"].is_monotonic_increasing}')
print(f'✓ Missing values   : {clean_aapl.isnull().sum().sum()}')
print(f'✓ Date range    : {clean_aapl["date"].min()} → {clean_aapl["date"].max()}')

## 2. Clean All Stocks

In [ ]:
# Load and clean all three stocks
all_stocks = load_and_clean_all_stocks(TICKERS, RAW_DIR)

print(f'Combined stock data: {all_stocks.shape}')
print(f'Tickers: {all_stocks["ticker"].unique()}')
print(f'\nRecords per ticker:')
print(all_stocks['ticker'].value_counts())

display(all_stocks.head())

In [ ]:
# Descriptive statistics after cleaning
print('Cleaned Stock Data — Statistics')
display(all_stocks.describe())

## 3. Mutual Fund Data Cleaning

In [ ]:
# Load raw mutual fund data
mf_raw = load_raw_mutual_funds(RAW_DIR)

print('BEFORE CLEANING')
print(f'Shape: {mf_raw.shape}')
print(f'Missing values:\n{mf_raw.isnull().sum()}')
display(mf_raw.head(3))

In [ ]:
# Apply mutual fund cleaning
mf_clean = clean_mutual_funds(mf_raw)

print('AFTER CLEANING')
print(f'Shape: {mf_clean.shape}')
print(f'Missing values:\n{mf_clean.isnull().sum()}')
print(f'\nCategories: {mf_clean["category"].nunique()}')
print(mf_clean['category'].value_counts())

display(mf_clean.head(5))

In [ ]:
# Mutual fund verification
print('Mutual Fund Cleaning Verification')
print('=' * 50)
print(f'✓ Columns lowercase  : {all(c == c.lower() for c in mf_clean.columns)}')
print(f'✓ returns_1yr is numeric : {pd.api.types.is_numeric_dtype(mf_clean["returns_1yr"])}')
print(f'✓ sd is numeric   : {pd.api.types.is_numeric_dtype(mf_clean["sd"])}')
print(f'✓ No missing scheme_name : {mf_clean["scheme_name"].notna().all()}')
print(f'✓ Categories standardized : {mf_clean["category"].str.istitle().all()}')

## 4. Export Cleaned Data

In [ ]:
# Save all cleaned datasets
os.makedirs(PROC_DIR, exist_ok=True)

# All stocks combined
save_processed(all_stocks, 'all_stocks_clean.csv', PROC_DIR)

# Individual stock files
for ticker in TICKERS:
 subset = all_stocks[all_stocks['ticker'] == ticker]
 save_processed(subset, f'{ticker}_clean.csv', PROC_DIR)

# Mutual funds
save_processed(mf_clean, 'mutual_funds_clean.csv', PROC_DIR)

print('\n All cleaned datasets saved to:', os.path.abspath(PROC_DIR))

In [ ]:
# Verify exported files
print('Processed files:')
for f in sorted(os.listdir(PROC_DIR)):
 if f.endswith('.csv'):
  size = os.path.getsize(os.path.join(PROC_DIR, f)) / 1024
  print(f' {f:30s} ({size:.1f} KB)')

## Summary

### Cleaning Operations Applied:
| Operation | Stock Data | Mutual Funds |
|-----------|-----------|-------------|
| Standardized column names | | |
| Removed '$' symbols | | N/A |
| Converted to numeric | | |
| Parsed dates to datetime | | N/A |
| Handled missing values | (ffill/bfill) | (median) |
| Added daily_return | | N/A |
| Sorted chronologically | | N/A |
| Standardized categories | N/A | (Title Case) |

### Next Steps
→ Proceed to **03_eda_analysis.ipynb** for exploratory data analysis